<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINTIA/blob/main/notebooks/02_gliner_entidades_pii.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 02 · Entidades e PII com GLiNER + validação brasileira

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINTIA)

O **GLiNER** extrai entidades de qualquer tipo que você nomear, sem treino (zero-shot), e roda até em CPU. O OSINTIA soma a ele validadores determinísticos brasileiros (CPF, CNPJ, DDD, IP público) e monta um **grafo de coocorrência** para análise de vínculos.

| | |
|---|---|
| **Hardware** | CPU (não precisa de GPU) |
| **Tempo estimado** | 5 minutos (CPU) · 1 minuto (GPU) |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
!pip install -q gliner

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINTIA.git /content/OSINTIA 2>/dev/null || git -C /content/OSINTIA pull -q
import sys
sys.path.insert(0, "/content/OSINTIA")
from osintia import auditoria, calibracao, custodia, dados, esquemas, extracao, roteamento
print("OSINTIA pronto")

## 1. Carregar o modelo
- `urchade/gliner_multi-v2.1`: multilíngue, bom em português, Apache-2.0
- `urchade/gliner_multi_pii-v1`: afinado para dados pessoais

In [ ]:
from gliner import GLiNER
MODELO = "urchade/gliner_multi-v2.1"
modelo = GLiNER.from_pretrained(MODELO)

## 2. Textos de exemplo (fictícios)

In [ ]:
textos = {
    "post_golpe": "Atenção clientes do Banco Aurora! Problemas com estorno? Fale com Marcos Vieira no WhatsApp (31) 99999-0000 ou mande e-mail para estorno@aurora-falso.example. Pix para a chave 529.982.247-25 libera em 1h.",
    "paste": "dump aurora_rh 2026: joana.prado@aurora.example ; carlos.m@aurora.example ; servidor 200.160.2.3 ; painel http://10.0.0.15/admin ; CNPJ 11.222.333/0001-81",
    "forum": "o @bancoaur0ra e o @suporte_bancoaurora sao do mesmo cara, usam a mesma carteira bc1qxy2kgdygjrsqtzq2n0yrf2493p83kkfjhx0wlh e postam de Belo Horizonte",
}
ROTULOS = ["pessoa", "organização", "local", "telefone", "e-mail", "username de rede social",
           "carteira de criptomoeda", "endereço IP", "URL", "documento CPF ou CNPJ"]

## 3. Extrair com GLiNER e confirmar com regras
O modelo propõe pelo contexto; a regra confirma o formato e o dígito verificador.

In [ ]:
MAPA = {"telefone": "telefone", "e-mail": "email", "endereço IP": "ip", "documento CPF ou CNPJ": ("cpf", "cnpj")}

def confirmar(rotulo, valor):
    alvo = MAPA.get(rotulo)
    if not alvo:
        return None  # sem regra para esse tipo: fica só a confiança do modelo
    alvos = alvo if isinstance(alvo, tuple) else (alvo,)
    regra = [e for e in extracao.extrair(valor) if e["tipo"] in alvos]
    return bool(regra) and all(e["valido"] for e in regra)

tabela = []
for doc, texto in textos.items():
    for ent in modelo.predict_entities(texto, ROTULOS, threshold=0.4):
        tabela.append({
            "doc": doc, "rotulo": ent["label"], "valor": ent["text"],
            "score": round(ent["score"], 3), "regra_confirma": confirmar(ent["label"], ent["text"]),
        })

for t in tabela:
    print(t)

## 4. O que o modelo deixou passar?
As regras rodam no texto inteiro. Tudo que a regra achou e o modelo não, vale revisar.

In [ ]:
for doc, texto in textos.items():
    achados_modelo = {t["valor"] for t in tabela if t["doc"] == doc}
    for e in extracao.extrair(texto):
        if not any(e["valor"] in v or v in e["valor"] for v in achados_modelo):
            print(doc, "->", e["tipo"], e["valor"], "| válido:", e["valido"])

## 5. Grafo de coocorrência
Entidades no mesmo documento ganham uma aresta. É a semente de uma análise de vínculos (exporte para Gephi, Maltego ou o VINCULA).

In [ ]:
import csv, itertools, os
from collections import Counter

arestas = Counter()
for doc in textos:
    ents = sorted({(t["rotulo"], t["valor"]) for t in tabela if t["doc"] == doc})
    for a, b in itertools.combinations(ents, 2):
        arestas[(a, b)] += 1

os.makedirs("/content/saidas", exist_ok=True)
with open("/content/saidas/vinculos.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["origem_tipo", "origem", "destino_tipo", "destino", "peso"])
    for (a, b), peso in arestas.items():
        w.writerow([a[0], a[1], b[0], b[1], peso])
print(len(arestas), "arestas em /content/saidas/vinculos.csv")

## 6. Relatório com LGPD
Para compartilhar, mascare identificadores pessoais e registre a cadeia de custódia.

In [ ]:
for doc, texto in textos.items():
    print(doc, "->", extracao.mascarar(texto))
custodia.registrar("/content/saidas/manifesto.jsonl", "extracao_gliner", "/content/saidas/vinculos.csv", {"modelo": MODELO})

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*